## Exploratory Data Analysis (EDA) Summary

**Scope:** Training set only (24,000 customers). The test set was not accessed.

| Finding | Decision |
|---|---|
| No missing values | No imputation required |
| EDUCATION codes 0, 5, and 6 are undocumented (289 rows) | Merged into category 4 (`Other`) |
| PAY_* default rates do not follow a linear pattern | One-hot encoding for logistic regression |
| PAY_0 values 5 to 8 have fewer than 20 customers each (value 4 has 58) | Rare categories in all PAY columns grouped using `min_frequency=100` |
| Monetary columns are right-skewed, with some negative bill amounts | Signed log transformation applied only within the logistic regression pipeline |
| BILL_AMT columns are highly correlated (0.80 to 0.95) | Regularization (`C`) in logistic regression |
| SEX, MARRIAGE, and AGE are sensitive features | Dropped during processing but retained in the CSV files for a later audit |

In [1]:
import pandas as pd

train = pd.read_csv("../data/processed/train.csv")
print(train.shape)
train.head()

(24000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,160000,2,2,2,33,2,2,3,2,0,...,164310,162681,163005,15000,0,0,6100,12300,6100,0
1,150000,2,1,2,34,1,-1,-1,-2,-2,...,0,0,0,53,0,0,0,0,0,0
2,10000,1,2,1,50,1,2,0,0,0,...,8067,8227,8400,2,1281,1134,294,305,1000,0
3,220000,2,1,2,29,0,0,0,0,0,...,218513,131660,134346,9100,9000,7887,4800,4900,6000,1
4,310000,2,1,2,32,1,-2,-1,0,0,...,326,-235,-235,0,326,0,0,0,1200,0


We have 24,000 customers in this specific chunk of data. For each customer, we have 24 pieces of information (like their age, education, past bills, etc.).

Default rate in the training set: 0.2212 (printed when the data was split).

Missing values.

In [2]:
train.isna().sum()

LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_0        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
default      0
dtype: int64

No missing values, so no imputation is needed.

EDUCATION and MARRIAGE.

In [3]:
print(train["EDUCATION"].value_counts().sort_index())
print(train["MARRIAGE"].value_counts().sort_index())

EDUCATION
0       12
1     8455
2    11256
3     3903
4       97
5      235
6       42
Name: count, dtype: int64
MARRIAGE
0       47
1    10892
2    12806
3      255
Name: count, dtype: int64


Codes 0, 5 and 6 are undocumented (289 rows). Code 4 is documented as "Other". High school (3) is about 16% of customers.

MARRIAGE: Most people are either Married (1) or Single (2).   

PAY columns.

In [4]:
print(train["PAY_0"].value_counts().sort_index())
print(train.groupby("PAY_0")["default"].agg(["mean", "count"]))

PAY_0
-2     2198
-1     4527
 0    11781
 1     3000
 2     2138
 3      245
 4       58
 5       18
 6       10
 7        9
 8       16
Name: count, dtype: int64
           mean  count
PAY_0                 
-2     0.126934   2198
-1     0.166335   4527
 0     0.128003  11781
 1     0.343667   3000
 2     0.693639   2138
 3     0.767347    245
 4     0.655172     58
 5     0.444444     18
 6     0.600000     10
 7     0.777778      9
 8     0.500000     16


Default rates by PAY_0: -2 is 12.7%, -1 is 16.6%, 0 is 12.8%, 1 is 34%, 2 is 69%. The meanings of -2 and 0 are not documented. About 69% of customers 2 months late miss the next payment.

In [5]:
print(train.groupby("PAY_2")["default"].agg(["mean", "count"]))

           mean  count
PAY_2                 
-2     0.179538   3030
-1     0.159752   4845
 0     0.159548  12573
 1     0.200000     25
 2     0.556121   3145
 3     0.637795    254
 4     0.500000     78
 5     0.619048     21
 6     0.727273     11
 7     0.529412     17
 8     0.000000      1


In [6]:
print(train.groupby("EDUCATION")["default"].agg(["mean", "count"]))

               mean  count
EDUCATION                 
0          0.000000     12
1          0.192904   8455
2          0.237562  11256
3          0.250064   3903
4          0.061856     97
5          0.063830    235
6          0.166667     42


In [7]:
print(train.groupby("SEX")["default"].agg(["mean", "count"]))

         mean  count
SEX                 
1    0.243728   9486
2    0.206490  14514


In [8]:
print(train.groupby("MARRIAGE")["default"].agg(["mean", "count"]))

              mean  count
MARRIAGE                 
0         0.106383     47
1         0.233015  10892
2         0.210370  12806
3         0.282353    255


For continuous columns like LIMIT_BAL and AGE, bin them first:

In [9]:
train.groupby(pd.qcut(train["LIMIT_BAL"], 5))["default"].mean()

LIMIT_BAL
(9999.999, 50000.0]      0.321673
(50000.0, 100000.0]      0.259230
(100000.0, 180000.0]     0.199512
(180000.0, 270000.0]     0.169178
(270000.0, 1000000.0]    0.130784
Name: default, dtype: float64

The lowest limits (under 50k) have a 32% default rate, the highest (over 270k) 13%.

Demographics: Men default slightly more than women. High school grads default slightly more than college grads. But these are tiny differences compared to their past payment history.

In [10]:
cols = ["LIMIT_BAL", "BILL_AMT1", "PAY_AMT1"]
print(train[cols].describe())
print("Negative BILL_AMT1:", (train["BILL_AMT1"] < 0).sum())

            LIMIT_BAL     BILL_AMT1       PAY_AMT1
count    24000.000000   24000.00000   24000.000000
mean    167364.666667   51100.50225    5623.556292
std     129511.313151   73510.03206   16148.316646
min      10000.000000 -165580.00000       0.000000
25%      50000.000000    3513.75000    1000.000000
50%     140000.000000   22330.00000    2100.500000
75%     240000.000000   66966.50000    5018.250000
max    1000000.000000  964511.00000  873552.000000
Negative BILL_AMT1: 468




468 customers have a negative BILL_AMT1. The data does not document why (possibly refunds or overpayments), so they were kept. The average credit limit is about 167,000 New Taiwan dollars.



Correlation.

In [11]:
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
train[bill_cols].corr().round(2)

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6
BILL_AMT1,1.00,0.95,0.90,0.86,0.83,0.80
BILL_AMT2,0.95,1.00,0.94,0.89,0.86,0.83
BILL_AMT3,0.90,0.94,1.00,0.93,0.89,0.86
BILL_AMT4,0.86,0.89,0.93,1.00,0.94,0.90
BILL_AMT5,0.83,0.86,0.89,0.94,1.00,0.95
BILL_AMT6,0.80,0.83,0.86,0.90,0.95,1.00


Test the "zero balance" idea for PAY_0 = -2

This tests our theory that -2 means an inactive account. If it is, the average bill amount for these people should be very close to 0.

In [12]:
print("--- BILL_AMT1 for PAY_0 = -2 ---")
print(train[train["PAY_0"] == -2]["BILL_AMT1"].describe())

--- BILL_AMT1 for PAY_0 = -2 ---
count      2198.000000
mean       8389.438581
std       24641.353076
min      -15308.000000
25%           0.000000
50%        1246.000000
75%        6092.000000
max      443118.000000
Name: BILL_AMT1, dtype: float64


Check for skewed financial data (Mean vs Median)

In [13]:
print("--- Skew Check (Mean vs Median) ---")
print(train[["LIMIT_BAL", "BILL_AMT1", "PAY_AMT1"]].describe().loc[["mean", "50%", "max"]])

--- Skew Check (Mean vs Median) ---
           LIMIT_BAL     BILL_AMT1       PAY_AMT1
mean   167364.666667   51100.50225    5623.556292
50%    140000.000000   22330.00000    2100.500000
max   1000000.000000  964511.00000  873552.000000


Mean is above median, so the money columns are right-skewed.

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

X_all = train.drop(columns="default")
for col in X_all.columns:
    tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42)
    auc = cross_val_score(tree, X_all[[col]], train["default"], cv=5, scoring="roc_auc").mean()
    print(f"{col:12s} {auc:.3f}")

LIMIT_BAL    0.623
SEX          0.526
EDUCATION    0.541
MARRIAGE     0.517
AGE          0.534
PAY_0        0.711
PAY_2        0.648
PAY_3        0.633
PAY_4        0.616
PAY_5        0.605
PAY_6        0.601
BILL_AMT1    0.527
BILL_AMT2    0.528
BILL_AMT3    0.523
BILL_AMT4    0.527
BILL_AMT5    0.536
BILL_AMT6    0.530
PAY_AMT1     0.611
PAY_AMT2     0.600
PAY_AMT3     0.594
PAY_AMT4     0.585
PAY_AMT5     0.580
PAY_AMT6     0.584


Each feature was scored alone with a shallow decision tree and 5-fold cross-validation. A weak feature alone can still help in combination with others.

In [15]:
import sys
sys.path.append("..")
from src.features import split_xy

X, y = split_xy(train)
print(X.shape)
print(X.columns.tolist())
print(X["EDUCATION"].value_counts().sort_index())

(24000, 20)
['LIMIT_BAL', 'EDUCATION', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
EDUCATION
1     8455
2    11256
3     3903
4      386
Name: count, dtype: int64
